# Newz — Phase 3 Calibration Notebook

Proves CLU-07 (staged demo clips fuse into one cluster) and CLU-08 (adversarial pair stays separate).

## Precondition

Backend MUST be running with `USE_MOCK_EMBEDDINGS=false` for true calibration.
Mock vectors are random and will not fuse — running this notebook with mock embeddings will FAIL CLU-07.

```bash
# In a separate terminal, with TWELVELABS_API_KEY in backend/.env:
cd backend && uvicorn app:app --port 8000
```

Set `BACKEND_URL` env var if backend runs on a different host/port.

In [1]:
import os, sys, json, time, subprocess
from pathlib import Path
import httpx

BASE = os.environ.get("BACKEND_URL", "http://localhost:8000")
print(f"calibrating against {BASE}")

calibrating against http://localhost:8000


In [2]:
r = httpx.get(f"{BASE}/health", timeout=5.0)
assert r.status_code == 200, f"backend not reachable at {BASE} (status {r.status_code})"
print("backend healthy")

backend healthy


In [3]:
# Run seed_demo.py from the repo root. Walk up from cwd until we find backend/seed/seed_demo.py
# so this works whether the notebook is launched from the repo root or backend/notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (p for p in candidates if (p / "backend" / "seed" / "seed_demo.py").exists()),
    None,
)
assert repo_root is not None, "could not locate repo root containing backend/seed/seed_demo.py"
print(f"running seed from {repo_root}")
result = subprocess.run(
    [sys.executable, "-m", "backend.seed.seed_demo", "--base-url", BASE],
    cwd=str(repo_root), check=True, capture_output=True, text=True,
)
print(result.stdout)
# Pipeline is fire-and-forget; wait long enough for embed (5-30s per clip) + cluster.
# 4 clips * ~25s/embed worst case = ~100s
time.sleep(60)

running seed from /Users/liamshalom/Hacktech


uploaded clip-1.mp4 -> clip_id=b76606cbcd094b0787c08e5824bb286b lat=34.13756 lng=-118.12544
uploaded clip-2.mp4 -> clip_id=745ef014255f419283487c66c62cdcbc lat=34.13770 lng=-118.12530
uploaded clip-3.mp4 -> clip_id=67d9d29950e24d179330891e72f770d3 lat=34.13784 lng=-118.12516
uploaded clip-4.mp4 -> clip_id=bb6a396c39154ecaaf6fa0188ee9ec3a lat=34.13797 lng=-118.12503
done. 4 clips uploaded. fetch /debug/clusters to inspect.



In [4]:
dbg = httpx.get(f"{BASE}/debug/clusters", timeout=10.0).json()
print(json.dumps(dbg, indent=2)[:2000])
print(f"...")
print(f"\ntotal clusters: {len(dbg['clusters'])}")
print(f"threshold: {dbg['threshold']}")
print(f"weights: {dbg['weights']}")

{
  "threshold": 0.55,
  "visual_floor": 0.8,
  "weights": {
    "visual": 0.55,
    "gps": 0.3,
    "time": 0.15
  },
  "gps_radius_m": 200.0,
  "time_window_s": 600.0,
  "clusters": [
    {
      "cluster_id": "402348d536614eb88c5944953f455a3e",
      "member_count": 17,
      "centroid_lat": 34.137747694753585,
      "centroid_lng": -118.1252523052464,
      "median_ts": 1777132444.3159194,
      "members": [
        {
          "clip_id": "b9bb3824191a4189a564123378e1395b",
          "lat": 34.13756486486487,
          "lng": -118.12543513513513,
          "ts": 1777131474.1983402,
          "visual": 0.9643,
          "gps": 0.868,
          "time": 0.0,
          "composite": 0.7908,
          "gps_available": true,
          "gps_distance_m": 26.4,
          "time_delta_s": 970.1
        },
        {
          "clip_id": "cdc12243db09461a89efb74931e54cdd",
          "lat": 34.1377,
          "lng": -118.1253,
          "ts": 1777131479.1983402,
          "visual": 0.9574,
      

In [5]:
# CLU-07: largest cluster must have >= 3 members
clusters = dbg["clusters"]
assert len(clusters) >= 1, "CLU-07 FAILED: no clusters formed at all"
biggest = max(clusters, key=lambda c: c["member_count"])
assert biggest["member_count"] >= 3, (
    f"CLU-07 FAILED: largest cluster has {biggest['member_count']} members, expected >=3.\n"
    f"  threshold={dbg['threshold']}, member breakdown:\n"
    f"  {json.dumps(biggest['members'], indent=2)}\n"
    f"  Try lowering CLUSTER_THRESHOLD env var or check Marengo cosine in the score breakdown."
)
print(f"CLU-07 PASS: {biggest['member_count']} clips fused into cluster {biggest['cluster_id'][:8]}...")
print(f"  composite scores per member: {[m['composite'] for m in biggest['members']]}")
print(f"  visual scores per member: {[m['visual'] for m in biggest['members']]}")
print(f"  gps distances (m): {[m['gps_distance_m'] for m in biggest['members']]}")
print(f"  time deltas (s): {[m['time_delta_s'] for m in biggest['members']]}")

CLU-07 PASS: 17 clips fused into cluster 402348d5...
  composite scores per member: [0.7908, 0.8163, 0.7981, 0.7831, 0.8834, 0.9102, 0.8932, 0.8795, 0.8754, 0.9442, 0.8354, 0.794, 0.7181, 0.7908, 0.8163, 0.7981, 0.7831]
  visual scores per member: [0.9643, 0.9574, 0.94, 0.966, 0.9643, 0.9574, 0.94, 0.966, 0.8408, 0.966, 0.8408, 0.8408, 0.7028, 0.9643, 0.9574, 0.94, 0.966]
  gps distances (m): [26.4, 6.9, 12.6, 32.1, 26.4, 6.9, 12.6, 32.1, 6.9, 6.9, 6.9, 6.9, 6.9, 26.4, 6.9, 12.6, 32.1]
  time deltas (s): [970.1, 965.1, 960.1, 955.1, 229.4, 224.4, 219.4, 214.4, 107.1, 107.1, 266.8, 432.5, 432.5, 947.6, 952.6, 957.6, 962.6]


In [6]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    members = biggest["members"]
    n = len(members)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    # Bar chart of per-member composite + visual
    labels = [f"clip-{i+1}" for i in range(n)]
    composites = [m["composite"] for m in members]
    visuals = [m["visual"] for m in members]
    x = np.arange(n)
    axes[0].bar(x - 0.2, composites, 0.4, label="composite")
    axes[0].bar(x + 0.2, visuals, 0.4, label="visual (cos)")
    axes[0].axhline(dbg["threshold"], color="red", linestyle="--", label=f"threshold={dbg['threshold']}")
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
    axes[0].set_ylabel("score")
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"cluster {biggest['cluster_id'][:8]}: per-member scores vs centroid")
    axes[0].legend()
    # GPS distance + time delta
    gps = [m["gps_distance_m"] or 0 for m in members]
    delt = [m["time_delta_s"] for m in members]
    axes[1].bar(x - 0.2, gps, 0.4, label="gps_distance_m")
    axes[1].bar(x + 0.2, delt, 0.4, label="time_delta_s")
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
    axes[1].set_ylabel("delta")
    axes[1].set_title("per-member geo + time deltas vs centroid")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig("calibration_breakdown.png", dpi=100)
    plt.show()
    print("saved calibration_breakdown.png")
except ImportError:
    print("matplotlib not installed; skipping visualization (text breakdown above is sufficient).")

matplotlib not installed; skipping visualization (text breakdown above is sufficient).


In [7]:
# CLU-08: adversarial pair must end up in DIFFERENT clusters
adv_dir = Path("../seed/demo")
adv1 = adv_dir / "adversarial-1.mp4"
adv2 = adv_dir / "adversarial-2.mp4"
if not (adv1.exists() and adv2.exists()):
    print("CLU-08 SKIPPED: adversarial-{1,2}.mp4 not present in backend/seed/demo/")
    print("  Record two unrelated clips (e.g., empty hallway + parking lot) and rerun this cell.")
else:
    # Upload both with same time + same place; assert they end up in DIFFERENT clusters
    now = time.time()
    def upload(path):
        with open(path, "rb") as f:
            files = {"file": (path.name, f.read(), "video/mp4")}
            data = {"lat": "34.1377", "lng": "-118.1253", "ts": str(now)}
            r = httpx.post(f"{BASE}/clips", files=files, data=data, timeout=30.0)
        r.raise_for_status()
        return r.json()["clip_id"]
    cid1 = upload(adv1); cid2 = upload(adv2)
    print(f"adversarial clip 1: {cid1}\nadversarial clip 2: {cid2}")
    time.sleep(60)
    dbg2 = httpx.get(f"{BASE}/debug/clusters", timeout=10.0).json()
    # Find which cluster each adversarial clip ended up in
    def cluster_of(clip_id, dbg):
        for c in dbg["clusters"]:
            if any(m["clip_id"] == clip_id for m in c["members"]):
                return c["cluster_id"]
        return None
    cl1 = cluster_of(cid1, dbg2); cl2 = cluster_of(cid2, dbg2)
    assert cl1 is not None and cl2 is not None, f"adversarial clips not assigned: {cl1}, {cl2}"
    assert cl1 != cl2, (
        f"CLU-08 FAILED: adversarial clips ended up in SAME cluster {cl1}.\n"
        f"  Threshold may be too low (current: {dbg2['threshold']}). Two unrelated visual scenes "
        f"should not fuse even at same time + same place."
    )
    print(f"CLU-08 PASS: adversarial clips in DIFFERENT clusters ({cl1[:8]}... vs {cl2[:8]}...)")

adversarial clip 1: 110d88ad0dad4ec7b8a50d7b32cd430c
adversarial clip 2: 7a7e13551b25499298fbfbbeadfbe898


CLU-08 PASS: adversarial clips in DIFFERENT clusters (621ef82a... vs 77094d15...)
